# Análise de features: olhar a tabela antes de modelar

Todo notebook deste repositório carrega a tabela, treina um baseline e mede. Este faz o passo que vem
**antes** disso e que costuma ser pulado: olhar o que os dados são.

Numa base de clientes, o que interessa saber antes de qualquer modelo é como cada coluna se distribui,
onde há valor faltante, quantas categorias cada coluna de texto tem, o que anda junto com o alvo, o que
está duplicado e, principalmente, **se alguma coluna já contém a resposta**.

O alvo é `cancelou`: a pergunta do negócio seria "quem vai cancelar?".


## Por que olhar antes de modelar

A tabela deste notebook tem defeitos plantados de propósito: valor faltante em uma coluna, linhas
duplicadas, uma coluna de texto com cardinalidade alta e uma coluna que entrega o alvo.

Nenhum desses defeitos aparece no `score` do modelo. Eles aparecem na conta que você faz, e é isso que
o notebook treina: a conta.


In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(context="notebook", style="whitegrid", palette="colorblind",
              rc={"figure.figsize": (9, 4)})

RANDOM_STATE = 42
ALVO = "cancelou"


def tabela_com_defeitos(n_linhas=900, seed=RANDOM_STATE):
    """Base sintética de clientes, com faltante, duplicata e uma coluna vazada."""
    rng = np.random.default_rng(seed)
    tabela = pd.DataFrame({
        "idade": np.clip(rng.normal(38, 12, n_linhas), 18, 90).round().astype(int),
        "renda_mensal": rng.lognormal(mean=8.2, sigma=0.6, size=n_linhas).round(2),
        "meses_de_casa": rng.integers(1, 96, n_linhas),
        "chamadas_suporte": rng.poisson(1.4, n_linhas),
        "plano": rng.choice(["basico", "padrao", "premium"], n_linhas, p=[0.5, 0.35, 0.15]),
        "cidade": rng.choice([f"cidade_{i:02d}" for i in range(1, 41)], n_linhas),
    })

    # o cancelamento depende de verdade de três colunas, e de nada além delas
    sinal = (0.55 * tabela["chamadas_suporte"]
             + 0.90 * (tabela["meses_de_casa"] < 12)
             - 0.0004 * tabela["renda_mensal"]
             + rng.normal(0, 1.0, n_linhas))
    tabela[ALVO] = (sinal > np.quantile(sinal, 0.82)).astype(int)

    # a coluna vazada: anotação do atendimento que, na prática, repete o alvo
    anotacao = tabela[ALVO].map({1: "informado no atendimento", 0: "nao consta"})
    trocados = rng.choice(n_linhas, size=int(0.08 * n_linhas), replace=False)
    anotacao.iloc[trocados] = np.where(
        anotacao.iloc[trocados] == "nao consta", "informado no atendimento", "nao consta")
    tabela["motivo_cancelamento"] = anotacao

    # faltante em uma coluna numérica e três linhas duplicadas, coladas no fim
    tabela.loc[rng.choice(n_linhas, size=60, replace=False), "renda_mensal"] = np.nan
    return pd.concat([tabela, tabela.iloc[[0, 7, 33]]], ignore_index=True)


tabela = tabela_com_defeitos()

print(f"tabela: {tabela.shape[0]} linhas x {tabela.shape[1]} colunas")
print(f"alvo: {tabela[ALVO].mean():.1%} de cancelamento")
print()
print(tabela.head(5).to_string())


## Como cada coluna se distribui

Antes de qualquer estatística, o desenho. Três perguntas por coluna numérica: onde está o centro, quão
espalhado é e a forma é simétrica?


In [ ]:
numericas = ["idade", "renda_mensal", "meses_de_casa", "chamadas_suporte"]

fig, eixos = plt.subplots(1, len(numericas), figsize=(16, 3.2))
for coluna, eixo in zip(numericas, eixos):
    sns.histplot(tabela[coluna], kde=True, ax=eixo)
    eixo.set_title(coluna)
    eixo.set_xlabel("")
plt.tight_layout()
plt.show()

resumo = tabela[numericas].describe().T[["mean", "50%", "std", "min", "max"]].round(2)
resumo.columns = ["media", "mediana", "desvio", "minimo", "maximo"]
display(resumo)
print("assimetria, em que 0 é simétrica e > 1 é cauda longa à direita:")
print(tabela[numericas].skew().round(2).to_string())


`renda_mensal` é o caso que ensina: média bem acima da mediana, com cauda longa à direita. Numa coluna
assim, a média não descreve ninguém, e um modelo que mede distância vai se incomodar mais do que o
necessário. É o tipo de coluna que pede transformação, assunto do notebook de engenharia de features.


## Valor faltante

Faltante não é zero. Zero é um valor medido; faltante é ausência de medição, e tratá-los igual inventa
dado. O primeiro passo é contar.


In [ ]:
faltantes = (tabela.isna().mean() * 100).round(2)
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)
display(faltantes.to_frame("faltantes (%)"))

linhas_afetadas = tabela.isna().any(axis=1).sum()
print(f"colunas com faltante: {len(faltantes)}")
print(f"linhas afetadas: {linhas_afetadas} de {len(tabela)} ({linhas_afetadas / len(tabela):.1%})")
print(f"se o caminho fosse apagar as linhas, sobrariam {len(tabela) - linhas_afetadas} linhas")


Com este volume, apagar a linha perde dado demais e apagar a coluna perde um preditor que tem relação
com o alvo. O caminho é imputar, e com uma condição: a imputação tem de acontecer **dentro** do
pipeline, depois da divisão entre treino e teste, senão a estatística do teste entra no treino. O
notebook de encoding mostra isso em código.


## Cardinalidade: quantas categorias cada coluna de texto tem

Cardinalidade é o número de valores distintos. É a medida que decide o custo do encoding: uma coluna de
três categorias vira duas colunas no one-hot, uma de quarenta categorias vira trinta e nove colunas
quase vazias.


In [ ]:
categoricas = tabela.select_dtypes(exclude="number").columns.tolist()
cardinalidade = pd.DataFrame({
    "valores_unicos": tabela[categoricas].nunique(),
    "razao_por_linha": (tabela[categoricas].nunique() / len(tabela)).round(3),
}).sort_values("valores_unicos", ascending=False)
display(cardinalidade)

sns.countplot(tabela, x="plano", order=tabela["plano"].value_counts().index)
plt.title("plano: poucas categorias, e uma delas com metade da base")
plt.show()

print("as dez cidades mais frequentes:")
print(tabela["cidade"].value_counts().head(10).to_string())


`plano` tem três categorias e uma delas concentra metade da base. `cidade` tem quarenta, com cerca de
vinte e três casos por categoria, em média. São dois problemas diferentes: a primeira pede one-hot, a
segunda pede outra estratégia
(agrupar categorias raras, ou codificar pela relação com o alvo), e é isso que o notebook de encoding
discute.


## O que anda junto com o alvo

Duas medidas de correlação, e a diferença entre elas importa. Pearson mede relação em linha reta.
Spearman mede ordem: se uma sobe, a outra sobe, sem exigir que seja em linha reta.


In [ ]:
sem_alvo = [c for c in tabela.select_dtypes(include="number").columns if c != ALVO]

relacao = pd.DataFrame({
    "pearson": tabela[sem_alvo + [ALVO]].corr()[ALVO].drop(ALVO),
    "spearman": tabela[sem_alvo + [ALVO]].corr(method="spearman")[ALVO].drop(ALVO),
})
relacao = relacao.reindex(relacao["pearson"].abs().sort_values(ascending=False).index)
display(relacao.round(3))

sns.barplot(x=relacao["pearson"], y=relacao.index)
plt.title("correlacao de Pearson com o alvo")
plt.xlabel("")
plt.show()


`renda_mensal` é o exemplo da diferença entre as duas medidas: Pearson dá -0,28 e Spearman dá -0,34, em
módulo. A relação é monótona (renda maior, menos cancelamento) mas não é uma reta, por causa da cauda
longa, e é isso que Pearson cobra. Spearman, que olha só a ordem, não cobra.

`meses_de_casa` ensina o caso oposto, e é a lição mais importante do notebook: as duas correlações
ficam perto de zero, e nem por isso a coluna é inútil. O efeito ali não é gradual, é um corte em doze
meses, e um corte numa coluna que vai de 1 a 95 é diluído por qualquer medida de correlação. A célula
abaixo mede o corte direto e mostra o tamanho do efeito que a correlação não viu.

A lição prática: correlação baixa não é prova de coluna inútil, é prova de que a relação não é linear
nem monótona no todo. Quem decide se a coluna entra é a medição em validação cruzada, não a correlação.


In [ ]:
sns.boxplot(tabela, x=ALVO, y="chamadas_suporte")
plt.title("chamadas ao suporte por desfecho")
plt.show()

sns.violinplot(tabela, x=ALVO, y="meses_de_casa")
plt.title("meses de casa por desfecho")
plt.show()

cedo = tabela["meses_de_casa"] < 12
print(f"cancela em {tabela.loc[cedo, ALVO].mean():.1%} de quem tem menos de 12 meses ({cedo.sum()} linhas)")
print(f"cancela em {tabela.loc[~cedo, ALVO].mean():.1%} dos demais ({(~cedo).sum()} linhas)")
print("o corte de doze meses aparece aqui e nao aparece na correlacao")

amostra = (tabela[["idade", "renda_mensal", "meses_de_casa", "chamadas_suporte", ALVO]]
           .dropna().sample(300, random_state=RANDOM_STATE))
sns.pairplot(amostra, hue=ALVO, diag_kind="kde", plot_kws={"alpha": 0.5, "s": 18})
plt.show()


## Duplicata

Linha duplicada não é erro de digitação, é erro de coleta: o mesmo registro entrou duas vezes no
extrato. O efeito é pior do que parece quando as duplicatas caem só de um lado do alvo, porque aí o
modelo aprende duas vezes a mesma história.


In [ ]:
duplicadas = tabela.duplicated().sum()
print(f"linhas duplicadas: {duplicadas} de {len(tabela)}")
if duplicadas:
    display(tabela[tabela.duplicated(keep=False)].sort_values(list(tabela.columns)).head(6))

sem_duplicata = tabela.drop_duplicates().reset_index(drop=True)
print(f"sem duplicata: {len(sem_duplicata)} linhas "
      f"({len(tabela) - len(sem_duplicata)} a menos, {1 - len(sem_duplicata) / len(tabela):.2%})")


## A coluna que entrega o alvo

Aqui está o defeito mais caro dos quatro. `motivo_cancelamento` é uma anotação feita no atendimento que
registra o cancelamento. Ela concorda com o alvo na maior parte das linhas, e por isso um modelo a
adoraria: acertaria muito sem aprender nada.

O problema é o tempo. Essa coluna só existe **depois** do cancelamento acontecer. No dia da previsão,
para um cliente que ainda não cancelou, ela não está preenchida. Uma coluna que só existe depois do
desfecho é vazamento: em treino ela dá nota alta, em produção ela não existe.


In [ ]:
confusao = pd.crosstab(tabela["motivo_cancelamento"], tabela[ALVO], margins=True)
display(confusao)

acordo = (tabela["motivo_cancelamento"].str.contains("informado") == tabela[ALVO].astype(bool)).mean()
print(f"a coluna concorda com o alvo em {acordo:.1%} das linhas")


def poder_de_adivinhar(tabela, alvo):
    """Quanto cada coluna sozinha acerta o alvo, pelo palpite mais comum de cada valor.

    Não é detector de vazamento, é termômetro: qualquer coluna que acerte bem mais do que a
    classe majoritária merece uma pergunta sobre quando ela é preenchida.
    """
    poder = {}
    for coluna in tabela.columns:
        if coluna == alvo:
            continue
        if tabela[coluna].dtype == object:
            palpite = tabela.groupby(coluna)[alvo].transform("mean").round().astype(int)
        else:
            palpite = (tabela[coluna] > tabela[coluna].median()).astype(int)
        poder[coluna] = (palpite == tabela[alvo]).mean()
    return pd.Series(poder).sort_values(ascending=False).round(3)


maioria = max(tabela[ALVO].mean(), 1 - tabela[ALVO].mean())
poder = poder_de_adivinhar(tabela, ALVO)
print(f"chutar sempre a classe majoritária acerta {maioria:.1%}")
display(poder.to_frame(f"acerta o alvo ({ALVO})"))


A coluna vazada aparece no topo, bem acima do palpite da classe majoritária, e é assim que ela se
denuncia: nenhuma outra coluna chega perto. Em dado de verdade, o vazamento não se chama
`motivo_cancelamento`, ele se chama `data_do_cancelamento`, `status_final`, `valor_pago` ou qualquer
coisa que só é preenchida depois que o evento que você quer prever aconteceu.

Regra prática: para cada coluna, pergunte **quando** ela passa a existir. Se a resposta for depois do
desfecho, ela não entra, por melhor que seja a correlação.


In [ ]:
sem_vazamento = tabela.drop(columns=["motivo_cancelamento"]).drop_duplicates().reset_index(drop=True)
matriz = sem_vazamento.select_dtypes(include="number").corr()

sns.heatmap(matriz, mask=np.triu(np.ones_like(matriz, dtype=bool), k=1), cmap="vlag", center=0,
            annot=True, fmt=".2f", annot_kws={"size": 7}, cbar_kws={"shrink": 0.8})
plt.title("correlacao entre as colunas numericas, depois de tirar a vazada")
plt.show()

print(f"tabela pronta para modelar: {sem_vazamento.shape[0]} linhas x {sem_vazamento.shape[1]} colunas")
print(f"colunas categoricas que ainda precisam de encoding: {len(categoricas) - 1}")
print(f"a imputacao de renda_mensal e a decisao sobre cidade continuam pendentes")


## Como ler isso

A lista abaixo é o que fazer com qualquer tabela nova, nesta ordem:

1. contar linhas, colunas e a proporção do alvo;
2. desenhar cada coluna numérica, e olhar a assimetria antes de olhar a média;
3. contar valor faltante por coluna, e nunca tratá-lo como zero;
4. contar categoria por coluna de texto, e perguntar o que o encoding vai custar;
5. medir correlação com o alvo em duas formas, sabendo que baixa não significa inútil;
6. procurar linha duplicada;
7. para cada coluna, perguntar **quando** ela passa a existir, e tirar as que só existem depois do
   desfecho.

Os itens 3, 4 e 7 são os que mais mudam o resultado, e são exatamente os que não aparecem no `score`.

## Resumo

Antes de escolher modelo, a tabela decide. Neste notebook: uma coluna com cauda longa que pede
transformação, 6,6% de faltante que pede imputação dentro do pipeline, uma coluna de texto com
quarenta categorias que pede encoding pensado, três linhas duplicadas que pedem deduplicação e uma
coluna vazada que pede remoção, por melhor que seja a correlação dela com o alvo.

E fica a lição que mais rende: a coluna que decide pode ter correlação perto de zero. `meses_de_casa`
tem 0,05 nas duas medidas e, mesmo assim, cancela 35,8% de quem tem menos de doze meses contra 15,5%
dos demais.

**Próximo passo:** o notebook de encoding desta pasta, onde essas decisões viram código, e depois o de
engenharia de features, onde a cauda longa da renda vira transformação.

## Referências

```bibtex
@book{tukey1977exploratory,
  title     = {Exploratory Data Analysis},
  author    = {Tukey, John W.},
  year      = {1977},
  publisher = {Addison-Wesley}
}

@article{kaufman2012leakage,
  title   = {Leakage in Data Mining: Formulation, Detection, and Avoidance},
  author  = {Kaufman, Shachar and Rosset, Saharon and Perlich, Claudia},
  journal = {ACM Transactions on Knowledge Discovery from Data},
  volume  = {6},
  number  = {4},
  pages   = {1--21},
  year    = {2012},
  doi     = {10.1145/2382577.2382579}
}
```
